In [1]:
from datasets import load_dataset, Dataset, concatenate_datasets
from trl import SFTConfig, SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback, DataCollatorWithPadding
import torch
from torch.utils.data import DataLoader
import random
from tqdm import tqdm
import re

/workspace/llm-autoformalization/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROMPT = """
Please autoformalize the following problem in Lean 4 with a header.
Use the following theorem names: my_favorite_theorem.
Your code should start with:
```Lean4
{header}
```
The natural language statement is:
{nl_problem}
"""

def prepare_dataset_lean_workbook(dataset, size=None):
    if size is None:
        size = len(dataset)

    header = "import Mathlib"

    data = []
    for sample in dataset:
        nl_problem = sample["natural_language_statement"]
        formal_problem = header + "\n\n" + sample["formal_statement"]
        formal_problem = re.sub(
            r"(theorem\s+)\w+",
            r"\1my_favorite_theorem",
            formal_problem
        )

        prompt = PROMPT.format(
            nl_problem=nl_problem,
            header=header
        )
        completion = f"```lean\n{formal_problem}\n```"

        data.append({
            "prompt": [{"role": "user", "content": prompt}],
            "completion": [{"role": "assistant", "content": completion}]
        })

    data = data[:size]
    
    return Dataset.from_list(data)

def get_header(lean_code):
    keywords = r"\b(theorem|def|lemma|variable|instance|structure|inductive)\b"

    # re.split разделит строку, и в элементе [0] будет всё, что шло ДО первого ключевого слова
    header_block = re.split(keywords, lean_code, maxsplit=1)[0]

    # Отрезаем лишние пробелы по краям для чистоты
    header_clean = header_block.strip()

    return header_clean

def prepare_dataset_formalmath(dataset, size=None):
    if size is None:
        size = len(dataset)

    data = []
    for sample in dataset:
        nl_problem = sample["refined_statement"]
        formal_problem = sample["autoformalization"]
        header = get_header(formal_problem)
        formal_problem = re.sub(
            r"(theorem\s+)\w+",
            r"\1my_favorite_theorem",
            formal_problem
        )

        prompt = PROMPT.format(
            nl_problem=nl_problem,
            header=header
        )
        completion = f"```lean\n{formal_problem}\n```"

        data.append({
            "prompt": [{"role": "user", "content": prompt}],
            "completion": [{"role": "assistant", "content": completion}]
        })

    data = data[:size]
    
    return Dataset.from_list(data)

In [3]:
dsname = "internlm/Lean-Workbook"
ds = load_dataset(dsname)
lean_workbook_dataset = prepare_dataset_lean_workbook(ds["train"])
lean_workbook_dataset.shape

(25214, 2)

In [4]:
dsname = "SphereLab/FormalMATH-All"
ds = load_dataset(dsname)["train"]
formalmath_dataset = prepare_dataset_formalmath(ds)
formalmath_dataset.shape

(5560, 2)

In [5]:
dataset = concatenate_datasets(
    [lean_workbook_dataset, formalmath_dataset]
)

dataset = dataset.shuffle(seed=42)

In [6]:
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

In [7]:
model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="cuda"
)


Loading weights: 100%|██████████| 434/434 [00:01<00:00, 276.40it/s]


In [8]:
config = SFTConfig(
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    warmup_steps=len(train_dataset)*0.05,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=800,
    save_strategy="epoch"
)

In [9]:
train = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=config
)

Tokenizing eval dataset: 100%|██████████| 3078/3078 [00:02<00:00, 1293.70 examples/s]


In [11]:
train.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
800,0.165561,0.166670,0.166745,770135.000000,0.947380
1600,0.134655,0.135149,0.117594,1525889.000000,0.956738
2400,0.104236,0.110653,0.112920,2290942.000000,0.963258
3200,0.093248,0.099100,0.096406,3053054.000000,0.967169
4000,0.094058,0.092729,0.090130,3805351.000000,0.969052
4800,0.086149,0.089177,0.083177,4574848.000000,0.970680
5600,0.095756,0.087859,0.084460,5335003.000000,0.971059
6400,0.079778,0.087689,0.084453,6096241.000000,0.971050
6924,0.086181,0.087747,0.084484,6591978.000000,0.971079


Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.53s/it]


TrainOutput(global_step=6924, training_loss=0.12425713970232809, metrics={'train_runtime': 2713.7517, 'train_samples_per_second': 10.206, 'train_steps_per_second': 2.551, 'total_flos': 1.5447989074771968e+17, 'train_loss': 0.12425713970232809, 'epoch': 1.0})

In [12]:
model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [13]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "left"

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id

In [ ]:
eval_dataset = load_dataset("PAug/ProofNetVerif")["test"]

In [15]:
eval_dataset[0]

{'id': 'Dummit-Foote|exercise_9_4_9',
 'nl_statement': 'Prove that the polynomial $x^{2}-\\sqrt{2}$ is irreducible over $\\mathbb{Z}[\\sqrt{2}]$. You may assume that $\\mathbb{Z}[\\sqrt{2}]$ is a U.F.D.',
 'lean4_src_header': 'import Mathlib\n\nopen Fintype Subgroup Set Polynomial Ideal\nopen scoped BigOperators\n\n',
 'lean4_formalization': 'theorem exercise_9_4_9 :\n  Irreducible (X^2 - C Zsqrtd.sqrtd : Polynomial (Zsqrtd 2)) :=',
 'lean4_prediction': 'theorem dummy (F : Type*) [Field F] {p : F[X]} (h_no_roots : ∀ a, p.eval a ≠ 0) : Irreducible p := sorry',
 'correct': False}

In [17]:
def get_prompt(sample):
    nl_problem = sample["nl_statement"]
    header = sample["lean4_src_header"]

    user_prompt = PROMPT.format(
        nl_problem=nl_problem,
        header=header
    )

    messages = [
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return {"prompt": text}

eval_dataset = eval_dataset.map(get_prompt)

Map: 100%|██████████| 1452/1452 [00:00<00:00, 14407.89 examples/s]


In [20]:
def tokenize_fn(example):
    return tokenizer(example["prompt"], truncation=True, max_length=512)

ds = eval_dataset.map(tokenize_fn)
ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

Map: 100%|██████████| 1452/1452 [00:00<00:00, 3255.61 examples/s]


In [21]:
collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

loader = DataLoader(
    ds,
    batch_size=16,
    shuffle=False,
    collate_fn=collator,
)

In [45]:
all_answers = []

device = "cuda" if torch.cuda.is_available() else "cpu"
for batch in tqdm(loader):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=256
        )

    generated_tokens = outputs[:, input_ids.shape[1]:].cpu()
    texts = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    all_answers.extend(texts)

100%|██████████| 91/91 [07:47<00:00,  5.14s/it]


In [87]:
import re

def extract_lean_code(text: str) -> str | None:
    match = re.search(r"```lean\n(.*?)```", text, re.DOTALL)

    if not match:
        return "bad theorem"

    return match.group(1).strip()


In [88]:
final_answers = [extract_lean_code(answer) for answer in all_answers]

In [90]:
eval_dataset = eval_dataset.add_column(
    "answers",
    final_answers
)

In [62]:
eval_dataset = eval_dataset.add_column(
    "raw_answers",
    all_answers
)

In [91]:
eval_dataset.to_csv("../benchmark_pred_finetuning.csv", index=False)

Creating CSV from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 38.35ba/s]


2386229